[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/hanging_chain/hanging_chain.ipynb)

# A hanging chain by Newton's method

A chain of beads joined by stiff springs hangs between two pins at the same height. At rest it takes the shape of least energy: the beads as low as gravity can pull them, the springs as little stretched as the beads' weight allows. The positions of all the beads together are one field, `Vector[beads]`, and the energy is one scalar function of it. Its derivative with respect to the field is a form that takes a displacement of every bead to the change in energy; its second derivative couples every bead with its neighbours. Solving the second against the first moves the whole chain at once, a Newton step, and a handful of them find the chain at rest to round-off. With stiff springs, its shape is close to the catenary, the curve of a hanging chain that cannot stretch.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import jax
import numpy as np
from scipy.optimize import brentq

from numga import Algebra, stack
from numga.backend.jax import JaxContext, derivative
from numga.sparse import SparseExtensor
from examples.animation import save_figure
from examples.mechanics.hanging_chain import render

jax.config.update("jax_enable_x64", True)

ga = Algebra("x+y+")
mv = JaxContext(ga, np.float64).multivector
BEADS = 31
LINKS = BEADS + 1
# The distance between the pins, and the length of the chain with its springs at rest.
SPAN = 2.0
LENGTH = 2.5
# Each spring's force per unit of strain, its stretch over its rest length.
STIFFNESS = 1e3
BEAD_MASS = 0.1
GRAVITY = 9.81
STEPS = 8
# The start: the beads evenly along a circular arc through the pins, each pin this angle from its lowest point.
START_TURN = 1.3
CURVE_SAMPLES = 200

## 1. The energy

Each spring stores energy in its stretch, half its stiffness over its rest length times the stretch squared, so that it pulls with the stiffness times its strain. Gravity's energy falls by the work the weight does as a bead moves: it is minus the scalar product of the weight with the bead's position, summed over the beads. Each link is its head less its tail, and its ends are beads or pins: link k runs from bead k - 1 to bead k, the first from the left pin and the last to the right one. Two sparse incidence maps say so, from the beads and from the pins to the links, each coupling a link to its ends by minus one at the tail and one at the head, so the links are a field computed from the field of beads in one line. The energies are summed over the sites of the links and of the beads. Leading batch axes stay independent chains, so the same function takes one chain or many.

The chain hangs in a plane, and its energy needs only the lengths of the links and the scalar product of each bead with its weight: the vectors of the plane, `x+y+`, measured from the point midway between the pins, hold everything it uses.

In [ ]:
left, right = mv.x * (-SPAN / 2), mv.x * (SPAN / 2)                             # [] Vector
pins = stack([left, right]).field()                                             # Vector[pins]
weight = mv.y * (-BEAD_MASS * GRAVITY)                                          # [] Vector
rest = LENGTH / LINKS
# Link k has bead k as its head, bead k - 1 as its tail, the left pin as the first link's tail and the
# right pin as the last link's head.
unit = mv.scalar(np.ones((BEADS, 1)))                                           # [beads] Scalar
heads = SparseExtensor(unit, np.arange(BEADS), np.arange(BEADS), (LINKS, BEADS))
tails = SparseExtensor(unit, np.arange(1, LINKS), np.arange(BEADS), (LINKS, BEADS))
bead_ends = heads - tails                                                       # [links, beads] Scalar
pin_ends = SparseExtensor(mv.scalar([[-1], [1]]), np.array([0, LINKS - 1]), np.arange(2), (LINKS, 2))   # [links, pins] Scalar


def energy(beads):
    """The springs' stretch energy less the work of the beads' weight, for every chain in the batch."""
    links = bead_ends * beads + pin_ends * pins                                 # [...] Vector[links]
    stretch = links.norm() - rest                                               # [...] Scalar[links]
    springs = STIFFNESS / (2 * rest) * stretch.squared()                        # [...] Scalar[links]
    return springs.batch().sum(axis=-1) - weight.scalar_product(beads).batch().sum(axis=-1)

## 2. A sagging start

Newton's method follows the curvature of the energy, and a spring's curvature is positive in every direction only while it is stretched: a compressed spring would rather buckle sideways. The chain starts with every spring stretched, its beads evenly along a circular arc through the pins, its links about eight percent longer than at rest. Each bead is the arc's lowest point turned about the arc's centre by a rotor, the angles evenly spaced between the two pins'.

In [ ]:
radius = SPAN / 2 / np.sin(START_TURN)
centre = mv.y * (radius * np.cos(START_TURN))                                   # [] Vector
angles = np.linspace(-START_TURN, START_TURN, LINKS + 1)[1:-1]                 # [beads]
start = (centre + (((mv.x ^ mv.y) * (-angles / 2)).exp() >> (mv.y * -radius))).field()   # [] Vector[beads]

## 3. Newton's method on the whole chain

The derivative of the energy with respect to the field is the gradient, `scalar <- vector[31]`: its step slot ranges over the beads, and binding a displacement of every bead gives the first-order change in energy. Its derivative in turn is the curvature, the bilinear form `scalar <- vector[31], vector[31]`, which pairs a displacement of every bead with another. A spring couples the two beads it joins, so the curvature's blocks vanish except between a bead and itself or its neighbours. Solving the curvature against the gradient finds the displacement of the whole chain that cancels the gradient to first order, in one coupled solve: the Newton step. The step is jitted, so that JAX traces the energy, both derivatives and the solve once and compiles them into one function.

In matrix notation this reads as the Hessian matrix $H$ of the energy in the $2 \cdot 31$ coordinates of the beads, block tridiagonal in $2 \times 2$ blocks, the vector $g$ of its partial derivatives, and the step $-H^{-1} g$.

In [ ]:
@jax.jit
def newton(beads):
    """One Newton step of the whole chain."""
    gradient = derivative(energy)(beads)                                        # [] Scalar <- Vector[beads]
    curvature = derivative(derivative(energy))(beads)                           # [] Scalar <- (Vector[beads], Vector[beads])
    return beads - curvature.solve(gradient)                                    # [] Vector[beads]


print(derivative(energy)(start))
print(derivative(derivative(energy))(start))

## 4. The iterates

From the arc, each step moves every bead at once. The gradient of all the iterates is taken in one call: their batch axis holds independent chains, each differentiated on its own. Its adjoint, bound to one, reads the gradient as a vector at every bead, the force the springs and gravity leave unbalanced there, which at rest vanishes. Measured in bead weights, it falls slowly while the chain is far from rest, and once below a weight, every step squares it: Newton's method converges quadratically.

In [ ]:
def newton_iterates(beads, steps: int):
    """The start and the chain after every Newton step."""
    yield beads
    for _ in range(steps):
        beads = newton(beads)
        yield beads


iterates = stack(newton_iterates(start, STEPS))                                 # [steps + 1] Vector[beads]
one = mv.scalar([1])
unbalanced = -jax.jit(derivative(energy))(iterates).adjoint()(one)              # [steps + 1] Vector[beads]
residual = unbalanced.norm() / (BEAD_MASS * GRAVITY)                            # [steps + 1] Scalar[beads]

## 5. The catenary

A hanging chain that cannot stretch takes the shape $y = a \cosh(x / a) + c$, where $a$ is its radius of curvature at the bottom. Its length between pins a span $s$ apart is $2 a \sinh(s / 2a)$, which fixes $a$ by the chain's length, and $c$ puts the pins at height zero. The springs stretch by one to two percent under this chain's weight, so the chain at rest hangs slightly below the catenary of its rest length, by a distance that shrinks with the stiffness.

In [ ]:
def catenary_scale(span: float, length: float) -> float:
    """The catenary's radius of curvature at the bottom: the root of 2 a sinh(span / 2a) == length."""
    return brentq(lambda scale: 2 * scale * np.sinh(span / (2 * scale)) - length, span / 20, 100 * span)


scale = catenary_scale(SPAN, LENGTH)
across = np.linspace(-SPAN / 2, SPAN / 2, CURVE_SAMPLES)                        # [samples]
catenary = mv.x * across + mv.y * (scale * (np.cosh(across / scale) - np.cosh(SPAN / (2 * scale))))   # [samples] Vector
save_figure(render.iterates(iterates, residual, catenary, left, right), "hanging_chain")

In [ ]:
# checks
# At rest every bead's unbalanced force vanishes to round-off against its weight; once below a weight,
# each Newton step squares it, in units of the weight; and the beads lie on the catenary of the rest
# length to within the stretch the chain's weight gives springs this stiff.
largest = residual.batch().to_array().max(axis=-1)                  # [steps + 1]
np.testing.assert_array_less(largest[-1], 1e-9)
converging = largest[:-1] < 1
np.testing.assert_array_less(largest[1:][converging], largest[:-1][converging] ** 2 + 1e-9)
sites = iterates[-1].batch()                                                    # [beads] Vector
height = (sites.scalar_product(mv.x) / scale).cosh() * scale - scale * np.cosh(SPAN / (2 * scale))   # [beads] Scalar
np.testing.assert_allclose(sites.scalar_product(mv.y).to_array(), height.to_array(), atol=LENGTH * BEADS * BEAD_MASS * GRAVITY / STIFFNESS)